# Apply CUPED and CUPAC independently before A/A and A/B tests

[All How-to Guides](README.md)

Prepare an adjusted metric once if you need it for several analyses. Run `CUPEDTransformer` and `CUPACExecutor` directly. Each returns an `ExperimentData` object with an adjusted column in `ds` and a variance reduction report. Pass the adjusted columns to `AATest` and `ABTest`.

The [CUPED and CUPAC tutorial](../quick_tutorials/CUPED_CUPAC.ipynb) explains the formulas and configuration through `ABTest`.

## Contents

- [Prepare the data](#manual-input)
- [Run CUPED](#manual-cuped)
- [Run CUPAC](#manual-cupac)
- [Check the overall mean](#manual-mean)
- [Check the metrics in A/A tests](#manual-aa)
- [Estimate the effect in A/B tests](#manual-ab)

<a id="manual-input"></a>

## Prepare the data

Use the same generator as in the main tutorial. For A/B, simulate full compliance with the randomized assignment `z` and construct `y` from the potential outcomes `y0` and `y1`. For A/A, take `y0`, the outcome under no treatment, and call it `y` in a separate dataset. With real A/A data, use a metric from a period before treatment.

CUPED uses `y_lag1` as its covariate. For CUPAC, supply two lags of the outcome and features. Roles use `parent` and `lag` to link the periods. `cofounders` names the underlying features used for prediction.

In [1]:
from IPython.display import display

from hypex import AATest, ABTest
from hypex.dataset import (
    Dataset,
    ExperimentData,
    FeatureRole,
    PreTargetRole,
    TargetRole,
    TreatmentRole,
)
from hypex.ml import CUPACExecutor
from hypex.reporters import CupacReporter, CupedReporter
from hypex.transformers import CUPEDTransformer
from hypex.utils.tutorial_data_creation import DataGenerator

generator = DataGenerator(
    n_samples=1_000,
    distributions={
        "X1": {"type": "normal", "mean": 1, "std": 1},
        "X2": {"type": "bernoulli", "p": 0.5},
        "y0": {"type": "normal", "mean": 1, "std": 5},
    },
    time_correlations={"X1": 0.2, "X2": 0.1, "y0": 0.8},
    effect_size=0.1,
    seed=42,
)
frame = generator.generate().rename(
    columns={"y0_lag_1": "y_lag1", "y0_lag_2": "y_lag2"}
)
frame["y"] = frame["y0"].where(frame["z"] == 0, frame["y1"])
metric_roles = {
    "y": TargetRole(cofounders=["X1", "X2"]),
    "y_lag1": PreTargetRole(parent="y", lag=1),
    "y_lag2": PreTargetRole(parent="y", lag=2),
    "X1_lag1": FeatureRole(parent="X1", lag=1),
    "X2_lag1": FeatureRole(parent="X2", lag=1),
    "X1_lag2": FeatureRole(parent="X1", lag=2),
    "X2_lag2": FeatureRole(parent="X2", lag=2),
}
ab_roles = {"z": TreatmentRole(), **metric_roles}
ab_input = ExperimentData(Dataset(roles=ab_roles, data=frame[list(ab_roles)]))
aa_input = ExperimentData(
    Dataset(roles=metric_roles, data=frame[list(metric_roles)].assign(y=frame["y0"]))
)
{"A/A": aa_input.ds.shape, "A/B": ab_input.ds.shape}

{'A/A': (1000, 7), 'A/B': (1000, 8)}

Each dataset has 1,000 observations. The additional `z` column in the A/B dataset holds the randomized assignment. The A/A test creates its own control and test groups.

<a id="manual-cuped"></a>

## Run CUPED

Pass a mapping from each metric to its historical covariate to `CUPEDTransformer` and execute it on `ExperimentData`. Give each method a copy of the input so you can reuse the originals. Display the adjusted values and report for the A/B data.

In [2]:
cuped = CUPEDTransformer(cuped_features={"y": "y_lag1"})
cuped_aa = cuped.execute(aa_input.copy())
cuped_ab = cuped.execute(ab_input.copy())

display(cuped_ab.ds.select(["y", "y_cuped"]).iloc[:3])
display(CupedReporter().report(cuped_ab))

,y,y_cuped
0,1.558344,0.857207
1,5.642941,4.288690
2,6.419831,6.155384


,feature,variance_reduction_pct
0,y_cuped,63.808189


`y_cuped` is available in `cuped_ab.ds` with `TargetRole`. `variance_reduction_pct` shows that variance fell by 63.8% in the A/B dataset. The adjusted column is ready for further analysis.

<a id="manual-cupac"></a>

## Run CUPAC

Choose `ridge` and set `random_state` for reproducible cross-validation. The executor trains the model to predict `y_lag1` from the outcome and features at lag 2. It then uses lag 1 to predict the current period. CUPAC centers the prediction across the full sample before applying the adjustment.

In [3]:
cupac = CUPACExecutor(cupac_models="ridge", random_state=42)
cupac_aa = cupac.execute(aa_input.copy())
cupac_ab = cupac.execute(ab_input.copy())

display(cupac_ab.ds.select(["y", "y_cupac"]).iloc[:3])
display(CupacReporter().report(cupac_ab)["variance_reductions"])

,y,y_cupac
0,1.558344,0.664577
1,5.642941,4.392953
2,6.419831,5.981459


,target,best_model,variance_reduction_cv,variance_reduction_real
0,y,ridge,64.062123,63.475523


`variance_reduction_cv` estimates variance reduction on historical data. `variance_reduction_real` measures it for the current outcome. The values are 64.1% and 63.5%, respectively. `y_cupac` is available in `cupac_ab.ds` with `AdditionalTargetRole`. Assign it `TargetRole` before passing it to a standalone test.

<a id="manual-mean"></a>

## Check the overall mean

Each method subtracts an adjustment centered across all observations. Compare the overall means of the original and adjusted A/B metrics.

In [4]:
{
    "y": ab_input.ds["y"].mean(),
    "y_cuped": cuped_ab.ds["y_cuped"].mean(),
    "y_cupac": cupac_ab.ds["y_cupac"].mean(),
}

{'y': 0.876109295951004,
 'y_cuped': 0.876109295951004,
 'y_cupac': 0.8761092959510038}

All three means are approximately 0.876109. The last digits differ due to floating-point rounding. Control and treatment means can change because the average adjustment within each group depends on chance imbalance in the covariate.

<a id="manual-aa"></a>

## Check the metrics in A/A tests

Pass one adjusted metric to each test, keeping the adjustment computed before the candidate splits. Run 200 splits per method with `dry_test=True`. Setting `n_iterations` overrides the default of 2,000 and gives both methods the same sequence of seeds.

In [5]:
aa_cuped_data = cuped_aa.ds.select("y_cuped")
aa_cupac_data = cupac_aa.ds.select("y_cupac").replace_roles({"y_cupac": TargetRole()})

aa_cuped = AATest(dry_test=True, n_iterations=200).execute(aa_cuped_data)
aa_cupac = AATest(dry_test=True, n_iterations=200).execute(aa_cupac_data)

display(aa_cuped.aa_score)
display(aa_cupac.aa_score)

,score,p-value,pass
y_cuped TTest test_1,0.95,0.212703,True
y_cuped KSTest test_1,0.95,NaN,True


,score,p-value,pass
y_cupac TTest test_1,0.95,0.201255,True
y_cupac KSTest test_1,0.95,NaN,True


In each `aa_score`, find the t-test row for the adjusted metric. Its `p-value` comes from a test of uniformity across the entire series of t-test p-values. Here it is about 0.213 for CUPED and 0.201 for CUPAC, both above 0.05. The [AATest tutorial](../quick_tutorials/AATest.ipynb#aa-p-value-plot) explains how to read the generated distribution plot.

<a id="manual-ab"></a>

## Estimate the effect in A/B tests

Keep `TreatmentRole` on `z` and pass one metric per run. Compare the original estimate with the CUPED and CUPAC estimates. `ABTest()` tests the metric it receives because the separate executors already applied the adjustment.

In [6]:
ab_baseline_data = ab_input.ds.select(["z", "y"])
ab_cuped_data = cuped_ab.ds.select(["z", "y_cuped"])
ab_cupac_data = cupac_ab.ds.select(["z", "y_cupac"]).replace_roles(
    {"y_cupac": TargetRole()}
)

display(ABTest().execute(ab_baseline_data).summary)
display(ABTest().execute(ab_cuped_data).summary)
display(ABTest().execute(ab_cupac_data).summary)

,feature,group,control mean,test mean,difference,difference %,ci lower,ci upper,TTest pass,TTest p-value
0,y,1,0.651179,1.103755,0.452575,69.500841,-0.15769,1.06284,NOT OK,0.146387


,feature,group,control mean,test mean,difference,difference %,ci lower,ci upper,TTest pass,TTest p-value
0,y_cuped,1,0.671685,1.083001,0.411316,61.236507,0.044636,0.777997,OK,0.028136


,feature,group,control mean,test mean,difference,difference %,ci lower,ci upper,TTest pass,TTest p-value
0,y_cupac,1,0.666001,1.088754,0.422753,63.476396,0.054444,0.791063,OK,0.024683


Compare `ci lower` and `ci upper` in the full reports. Both adjusted metrics yield narrower confidence intervals. The t-test p-values are about 0.146 for `y`, 0.028 for `y_cuped`, and 0.025 for `y_cupac`.